# 03 — Metrics and fail-closed safety

Numbers below are **internal synthetic complete-case checks**.
They are not IPCW clinical performance and not CEC-adjudicated BVF discrimination.


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent):
    if (_candidate / "_paths.py").is_file():
        _nb = str(_candidate.resolve())
        if _nb not in sys.path:
            sys.path.insert(0, _nb)
        break
else:
    raise RuntimeError("Open this notebook from the ValveGuard repo (root or notebooks/).")

import json

from _paths import ARTIFACT_DIR
from backend.runtime import ValveGuardRuntime
from backend.tests.fixtures import presentation_demo_payloads
from model.tier1_varc3_rules import ClinicalContext

hist = json.loads((ARTIFACT_DIR / "training_history.json").read_text(encoding="utf-8"))
metrics = hist["metrics"]["tier2_first_event"]
print("metric_note", metrics["1_year"]["metric_note"])
for horizon in ("1_year", "3_year", "5_year", "10_year"):
    block = metrics[horizon]
    print(
        horizon,
        "auc",
        round(block["calibrated_auc_complete_case"], 3),
        "events",
        block["positive_events"],
        "n",
        block["evaluable_patients"],
    )

print("omitted flag stays", ClinicalContext.from_mapping({}).suspected_thrombosis)
try:
    ClinicalContext.from_mapping({"known_nsvd": False})
    raise SystemExit("known_nsvd should be rejected")
except ValueError as exc:
    print("known_nsvd rejected:", exc)

runtime = ValveGuardRuntime.load(ARTIFACT_DIR)
stage3 = runtime.predict(presentation_demo_payloads()["stage3_fail_closed"])
print("stage3 status", stage3["engine_status"], "tier2", stage3["tier2_first_event"])
assert stage3["engine_status"] == "abstained"
assert stage3["tier2_first_event"] is None
print("fail-closed ok")


Policy thresholds are **draft**. The code will not apply numeric cutoffs
until a clinician group is recorded. `intervention_recommendation` is always `null`.
